# Excepciones

El **flujo de control** es el orden en que se ejecutan las instrucciones de un programa: de arriba abajo, salvo cuando un condicional (`if`), un bucle (`for`, `while`) o la llamada a una función cambian ese orden.

Las **excepciones** son eventos que interrumpen ese flujo. Python las lanza automáticamente cuando se produce un error, pero el código también puede lanzarlas e interceptarlas de forma deliberada.

Por ejemplo, en España es habitual escribir los decimales con coma. Si un programa intenta convertir ese texto en número, Python lanza una excepción y detiene la ejecución:

```
float("40,4168")
ValueError: could not convert string to float: '40,4168'
```

Este cuaderno presenta las sentencias relacionadas con las excepciones:

* `try/except`: captura y gestiona las excepciones lanzadas por Python o por el propio código.
* `try/finally`: ejecuta acciones de limpieza, se produzca o no una excepción.
* `raise`: lanza una excepción manualmente.
* `assert`: lanza una excepción si una condición no se cumple.
* `with/as`: gestiona recursos, como archivos, y garantiza su liberación automática.

## Excepciones frecuentes

| Excepción | Cuándo se produce | Ejemplo |
|---|---|---|
| `ValueError` | El tipo es correcto, pero el contenido no es válido | `float("40,4168")` |
| `TypeError` | Operación con un tipo inadecuado | `"40.4" + 1` |
| `KeyError` | Clave inexistente en un diccionario | `crs_names[9999]` |
| `IndexError` | Posición fuera de rango en una lista | `[40.4, -3.7][5]` |
| `ZeroDivisionError` | División entre cero | `area / 0` |
| `FileNotFoundError` | El archivo no existe | `open("missing.shp")` |

## La sentencia `try/except`

Estructura general:

```
try:
    statements                 # Se ejecuta primero
except ExceptionA:
    statements                 # Se ejecuta si ExceptionA se lanza en el bloque try
except (ExceptionB, ExceptionC):
    statements                 # Se ejecuta si se lanza cualquiera de las dos
except ExceptionD as var:
    statements                 # Igual, pero la excepción queda guardada en var
else:
    statements                 # Se ejecuta solo si NO hubo excepción
finally:
    statements                 # Se ejecuta siempre
```

Reglas importantes:

* Los bloques `except` se evalúan **en orden** y solo se ejecuta el primero que coincide. Por ello se escriben primero las excepciones más específicas.
* Existe un `except:` sin nombre que captura cualquier excepción, pero se desaconseja porque oculta errores inesperados. Es preferible indicar siempre qué excepciones se esperan.

In [ ]:
# Converting coordinate texts into numbers
for text in ["40.4168", "40,4168", None]:
    try:
        latitude = float(text)
    except ValueError as error:
        print(f"Invalid value: {error}")
    except TypeError as error:
        print(f"Wrong type: {error}")
    else:
        print(f"Latitude: {latitude}")

En el siguiente ejemplo se busca el nombre de un sistema de referencia de coordenadas (CRS) a partir de su código EPSG. El bloque `else` solo se ejecuta si la búsqueda tiene éxito, y `finally` se ejecuta siempre.

In [ ]:
crs_names = {
    4326: "WGS 84",
    25830: "ETRS89 / UTM zone 30N",
    3857: "WGS 84 / Pseudo-Mercator",
}


def describe_crs(epsg_code):
    """Print the name of a CRS from its EPSG code."""
    try:
        name = crs_names[epsg_code]
    except KeyError:
        print(f"EPSG:{epsg_code} is not in the catalogue")
    else:
        print(f"EPSG:{epsg_code} is {name}")
    finally:
        print("Lookup finished\n")


describe_crs(25830)
describe_crs(9999)

## La sentencia `try/finally`

La cláusula `finally` se ejecuta siempre al salir del bloque `try`, tanto si se produjo una excepción como si no. Se utiliza para tareas de limpieza, como cerrar archivos o conexiones.

```
try:
    statements     # Se ejecuta primero
finally:
    statements     # Se ejecuta siempre al salir
```

`try/finally` **no captura** la excepción: tras ejecutar el bloque `finally`, la excepción sigue propagándose. Por eso, en el ejemplo, se combina con un `try/except` exterior.

In [ ]:
def mean_elevation(elevations):
    """Return the mean of a list of elevations."""
    try:
        return sum(elevations) / len(elevations)
    finally:
        print("Calculation attempted")


try:
    mean_elevation([])
except ZeroDivisionError:
    print("Empty list: the mean cannot be computed")

## La sentencia `raise`

`raise` lanza una excepción de forma manual. Es útil para rechazar datos que Python considera válidos pero que no tienen sentido en el contexto del problema, como una latitud de 95°.

In [ ]:
def check_latitude(latitude):
    """Return the latitude, or raise ValueError if it is out of range."""
    if latitude < -90 or latitude > 90:
        raise ValueError(f"Latitude out of range: {latitude}")
    return latitude


try:
    check_latitude(95)
except ValueError as error:
    print(error)

## La sentencia `assert`

`assert condición, mensaje` lanza un `AssertionError` si la condición es falsa. Se usa para comprobar suposiciones internas del programa durante el desarrollo.

No debe emplearse para validar datos externos (por ejemplo, los introducidos por un usuario), porque Python puede ejecutarse con las aserciones desactivadas (opción `-O`). Para eso se utiliza `raise`.

In [ ]:
def hectares_to_km2(hectares):
    """Convert hectares to square kilometres."""
    assert hectares >= 0, "Area cannot be negative"
    return hectares / 100


print(hectares_to_km2(250))

try:
    hectares_to_km2(-5)
except AssertionError as error:
    print(f"Assertion failed: {error}")

## Gestores de contexto: `with/as`

En el cuaderno `08_import_open.ipynb` se usó `with` para abrir archivos con la garantía de que se cierran al terminar el bloque. Con lo visto en este cuaderno se puede precisar qué hace: **`with` equivale a un `try/finally` automático**.

Las dos versiones de la siguiente celda son equivalentes. En la primera, `finally` cierra el archivo con `.close()` aunque se produzca un error al leerlo; en la segunda, `with` lo hace sin escribirlo. El método `.readline()` lee una sola línea, y el atributo `.closed` indica si el archivo está cerrado:

In [ ]:
from pathlib import Path

stations_path = Path.cwd().parent / "data" / "weather_stations.csv"

# Manual version: try/finally guarantees that the file is closed.
file = open(stations_path, encoding="utf-8")
try:
    header = file.readline()
finally:
    file.close()

print(header.strip())
print(file.closed)

# Equivalent version: with closes the file automatically.
with open(stations_path, encoding="utf-8") as file:
    header = file.readline()

print(header.strip())
print(file.closed)

Los objetos que funcionan con `with` se llaman **gestores de contexto** (*context managers*): saben prepararse antes de un bloque de código y limpiarse después, incluso si se produce una excepción. Los archivos son el caso más habitual, pero no el único; la sección *Aspectos avanzados* muestra cómo crear uno propio.

También se pueden usar varios gestores de contexto en una misma sentencia, separados por comas. Si la línea es larga, se pueden agrupar entre paréntesis:

```python
with (
    open(input_path, encoding="utf-8") as source,
    open(output_path, "w", encoding="utf-8") as target,
):
    ...
```

## Excepciones definidas por el usuario

Se pueden crear excepciones propias definiendo una clase que **herede** de `Exception`, como se vio en el cuaderno `09_classes_oop.ipynb`. Así, los errores específicos del dominio, como una coordenada no válida, tienen un nombre descriptivo y se pueden capturar de forma selectiva.

La clase puede guardar información adicional. `super().__init__(...)` pasa el mensaje a `Exception`, que lo mostrará al imprimir el error.

In [ ]:
class InvalidCoordinateError(Exception):
    """Raised when a latitude or longitude is out of range."""

    def __init__(self, axis, value):
        super().__init__(f"Invalid {axis}: {value}")
        self.axis = axis
        self.value = value


def validate_point(latitude, longitude):
    """Return (latitude, longitude), or raise InvalidCoordinateError."""
    if latitude < -90 or latitude > 90:
        raise InvalidCoordinateError("latitude", latitude)
    if longitude < -180 or longitude > 180:
        raise InvalidCoordinateError("longitude", longitude)
    return (latitude, longitude)


try:
    validate_point(40.4, 200)
except InvalidCoordinateError as error:
    print(error)
    print(f"Offending axis: {error.axis}, value: {error.value}")

## Ejercicios

### Coordenadas con coma decimal

Definir la función `parse_coordinate(text)`, que convierte un texto en un número decimal. Debe aceptar tanto punto como coma decimal (`"40.4168"` y `"40,4168"`) y devolver `None` si el texto no representa un número (por ejemplo, `"abc"`).

*Pista:* el método `.replace(",", ".")` de los textos sustituye la coma por un punto.

### Leer la primera línea de un archivo

Definir la función `read_first_line(path)`, que devuelve la primera línea de un archivo usando `with`. Si el archivo no existe, debe capturar `FileNotFoundError` y devolver el texto `"File not found"`.

Probarla con la ruta a `weather_stations.csv`, en la carpeta `data`, y con la ruta a un archivo inexistente, como `missing.csv`.

### Centroide de un conjunto de puntos

Definir la excepción `EmptyGeometryError` y la función `compute_centroid(points)`, que recibe una lista de tuplas `(latitude, longitude)` y devuelve la media de las latitudes y la media de las longitudes. Si la lista está vacía, debe lanzar `EmptyGeometryError` con un mensaje descriptivo.

Llamar a la función con una lista vacía dentro de un `try/except` que imprima el mensaje.

## Aspectos avanzados

Esta sección explica cómo funciona internamente un gestor de contexto. No es necesaria para usar `with` con archivos, pero ayuda a entender el código de muchas bibliotecas.

### Funcionamiento de un gestor de contexto

1. Se evalúa la expresión, que produce el gestor de contexto: un objeto con los métodos `__enter__` y `__exit__`.
2. Se llama a `__enter__`. Su valor de retorno se asigna a la variable de `as`, si existe.
3. Se ejecuta el bloque `with`.
4. Si el bloque lanza una excepción, se llama a `__exit__(tipo, valor, traceback)` con los detalles del error. Si devuelve un valor falso, la excepción se **vuelve a lanzar** (lo habitual); si devuelve un valor verdadero, se suprime.
5. Si no hay excepción, `__exit__` se llama igualmente, con los tres argumentos a `None`.

#### Gestor de contexto propio

Cualquier clase con `__enter__` y `__exit__` es un gestor de contexto. La siguiente registra el inicio y el fin de un paso de procesamiento.

In [ ]:
class ProcessingStep:
    """Print messages when a processing step starts and ends."""

    def __init__(self, name):
        self.name = name

    def __enter__(self):
        print(f"Starting: {self.name}")
        return self

    def __exit__(self, exc_type, exc_value, exc_tb):
        if exc_type is None:
            print(f"Finished: {self.name}\n")
        else:
            print(f"Failed: {self.name} ({exc_type.__name__})")
        return False  # Propagate the exception, if any

In [ ]:
# Without exceptions
with ProcessingStep("reproject layer") as step:
    print(f"Working on {step.name}...")

In [ ]:
# With an exception: __exit__ runs first, then the exception propagates
try:
    with ProcessingStep("read layer"):
        raise ValueError("Corrupted geometry")
        print("This line is never reached")
except ValueError as error:
    print(f"Caught outside the with block: {error}")

## Resumen

- El **flujo de control** es el orden en que se ejecutan las instrucciones. Una **excepción** lo interrumpe cuando se produce un error.
- `try/except` captura excepciones concretas. Los bloques `except` se evalúan en orden, por lo que las excepciones más específicas se escriben primero. Conviene evitar `except:` sin nombre.
- `else` se ejecuta solo si no hubo excepción; `finally`, siempre.
- `try/finally` no captura la excepción: ejecuta la limpieza y la excepción sigue propagándose.
- `raise` lanza una excepción manualmente, por ejemplo, para rechazar datos fuera de rango.
- `assert` comprueba suposiciones internas durante el desarrollo; no debe usarse para validar datos externos.
- `with` equivale a un `try/finally` automático: el recurso, como un archivo, se libera al terminar el bloque. Los objetos que lo permiten se llaman **gestores de contexto**.
- Las excepciones propias se definen con una clase que hereda de `Exception`.